### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants"

In [2]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [6]:
#Step 1: Load documents and split the dataset into chunks
loader = TextLoader("langchain_crewai_dataset.txt")
raw_documents = loader.load()

# splitting the document into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 40)
chunks = splitter.split_documents(raw_documents)

In [7]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standa

In [ ]:
# Step 2 : Creating Vector store
embedding_model = HuggingFaceEmbeddings(model = "all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Step 3 : MMR retriever
retriever = vector_store.as_retriever(search_typre = "mmr" , search_kwargs = {"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002105B4B4830>, search_kwargs={'k': 5})

In [ ]:
# Step : 4 LLM & prompt 
import os
from dotenv import load_dotenv
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

llm = init_chat_model("openai:gpt-5-nano-2025-08-07")
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002105D1DE3C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002105D1DEE40>, root_client=<openai.OpenAI object at 0x000002105CE03770>, root_async_client=<openai.AsyncOpenAI object at 0x000002105D1DEBA0>, model_name='gpt-5-nano-2025-08-07', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [15]:
# Query Expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
"""
)

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002105D1DE3C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002105D1DEE40>, root_client=<openai.OpenAI object at 0x000002105CE03770>, root_async_client=<openai.AsyncOpenAI object at 0x000002105D1DEBA0>, model_name='gpt-5-nano-2025-08-07', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)
| StrOutputParser()

In [16]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'Expanded query:\n"LangChain memory" OR "LangChain conversation memory" OR "LangChain memory module" OR "LangChain BaseMemory" OR "ConversationBufferMemory" OR "ConversationBufferWindowMemory" OR "ConversationSummaryMemory" OR "memory persistence" OR "persistent memory" OR "short-term memory" OR "long-term memory" OR "state management" OR "context retention" OR "memory backends" OR "memory adapters" OR "database-backed memory" OR "Redis" OR "SQLite" OR "PostgreSQL" OR "vector store with memory" OR "Retrieval Augmented Generation" OR "RAG"'

In [17]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [20]:
# Step 5 : Full RAG pipeline with Query expansion
rag_pipeline = (
    RunnableMap({
        "input" : lambda x : x["input"],
        "context" : lambda x : retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [21]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:
- LangChain memory types and modules: What kinds of memory are supported (buffer-based memory, summarization-based memory, and entity-based memory), and how to choose among them for chat applications? Include specific implementations such as ConversationBufferMemory, ConversationSummaryMemory, and ConversationEntityMemory, their use cases, limitations, and relevant APIs (e.g., save_context, load_memory_variables), plus guidance on when to use each.
- LangChain memory interfaces and persistence: What is the Memory interface in LangChain, what methods does it expose, and how do memory backends differ (ephemeral vs. persistent memory)? Cover integration with storage backends (e.g., Redis, MongoDB, SQLite/file-based stores), data serialization formats, and patterns for long-running conversations.
- Comparison of LangChain memory options: Ephemeral vs. persistent memory, behavior during long dialogues, memory retention policies, and how memory affects prompt history, latency

In [22]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:
{"input": "(CrewAI OR 'Crew AI' OR 'crew AI agents' OR 'AI agents for crews' OR 'intelligent crew management' OR 'autonomous crew coordination' OR 'autonomous agents' OR 'multi-agent systems' OR 'agent-based systems' OR 'BDI agents' OR 'Belief-Desire-Intention' OR 'planning-based agents' OR 'cognitive agents' OR 'task allocation' OR 'task scheduling' OR 'crew scheduling' OR 'rostering' OR 'resource allocation' OR 'workflow automation' OR 'robotic process automation' OR 'RPA' OR 'AI planning' OR 'crew optimization' OR 'airline crew scheduling' OR 'aviation crew management' OR 'film production crew management' OR 'maritime crew management')"}
✅ Answer:
 CrewAI agents are semi-independent autonomous agents that operate within a crew, each with a defined role (such as researcher, planner, or executor) and capable of forming structured workflows in a collaborative context. They collaborate to accomplish goals by distributing tasks across these roles.


In [23]:
# Step 6: Run query
query = {"input": " What type of memory CrewAI agents support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:
{"input": "What types of memory do CrewAI agents support? Include memory modalities, architectures, and persistence options such as volatile memory (RAM/DRAM) and non-volatile/persistent memory (SSD/NVMe, PCM, MRAM, persistent memory like 3D XPoint), working memory, short-term vs long-term memory, episodic memory, semantic memory, scratchpad memory, external memory stores (vector databases, knowledge bases, document stores), vector stores and embedding-based retrieval, explicit vs implicit memory models, episodic vs semantic memory distinctions, context windows and token-level memory, memory slots and memory management (read/write/update operations, eviction/policies), persistence and versioning, stateless vs stateful operation, interoperability with databases and knowledge graphs, integrations with ML/RAG tooling (PyTorch, TensorFlow, LangChain, LlamaIndex, Haystack), performance considerations (latency, throughput, memory footprint), scalability, security and privacy 